In [0]:
df = spark.read.format("csv").option("inferSchema", True).option("header", True).load('/Volumes/workspace/marts/datas/BigMart Sales.csv')

In [0]:
from pyspark.sql.functions import *
df.dropDuplicates(subset=["Item_Type"]).limit(5).display()

Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
FDP49,9.0,Regular,0.069088961,Breakfast,56.3614,OUT046,1997,Small,Tier 1,Supermarket Type1,1547.3192
DRI11,null,Low Fat,0.034237682,Hard Drinks,113.2834,OUT027,1985,Medium,Tier 3,Supermarket Type3,2303.668
FDH35,18.25,Low Fat,0.0,Starchy Foods,164.7526,OUT045,2002,null,Tier 2,Supermarket Type1,4604.6728
FDC14,null,Regular,0.072221801,Canned,43.6454,OUT019,1985,Small,Tier 1,Grocery Store,125.8362
FDX07,19.2,Regular,0.0,Fruits and Vegetables,182.095,OUT010,1998,null,Tier 3,Grocery Store,732.38


In [0]:
data1 = [('1', 'kad'), ('2', 'sid')]
schema1 = 'id STRING, name STRING'

df1 = spark.createDataFrame(data1, schema1)

data2 = [('3', 'rahul'), ('4', 'jas')]
schema2 = 'id STRING, name STRING'

df2 = spark.createDataFrame(data2, schema2)

df1.union(df2).display()  #o unionByName faz a mesma coisa, porém identifica o nome das colunas, e nao faz a uniao somente por ordem

id,name
1,kad
2,sid
3,rahul
4,jas


In [0]:
df.select(initcap(col('Item_Type'))).display()

initcap(Item_Type)
Dairy
Soft Drinks
Meat
Fruits And Vegetables
Household
Baking Goods
Snack Foods
Snack Foods
Frozen Foods
Frozen Foods


In [0]:
d = df.withColumn('current_date', current_date()) # mostra a data atual
d = df.withColumn('add_date', date_add(col('current_date'), 30)) # soma 30 dias na data atual
d = df.withColumn('sub_date', date_sub(col('current_date'), 30)) # subtrai 30 dias na data atual
d = df.withColumn('add_interval', add_months(col('current_date'), 3)) # soma 3 meses na data atual

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8366132554736377>, line 5
      3 d = df.withColumn('sub_date', date_sub(col('current_date'), 30)) # subtrai 30 dias na data atual
      4 d = df.withColumn('add_interval', add_months(col('current_date'), 3)) # soma 3 meses na data atual
----> 5 d.display()

File /databricks/python_shell/lib/dbruntime/monkey_patches.py:100, in apply_dataframe_display_patch.<locals>.df_display(df, *args, **kwargs)
     96 def df_display(df, *args, **kwargs):
     97     """
     98     df.display() is an alias for display(df). Run help(display) for more information.
     99     """
--> 100     display(df, *args, **kwargs)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     s

In [0]:
df.dropna('all') #exclui se a linha tiver todos os valores nulos
df.dropna('any') #exclui se a linha tiver pelo menos um valor nulo
df.dropna('all', subset=['Item_Weight', 'Item_Fat_Content']) #exclui se a linha tiver todos os valores nulos na coluna Item_Weight e na coluna Item_Fat_Content
df.dropna('any', subset=['Item_Weight', 'Item_Fat_Content']) #  exclui se a linha tiver pelo menos um valor nulo na coluna Item_Weight e na coluna Item_Fat_Content
df.fillna('nao disponivel') #substitui todos os valores nulos

In [0]:
df.withColumn('Outlet_Type', split(col('Outlet_Type'), ' ')) # tranforma valores em uma array, o espaço sendo delimitador
df.withColumn('Outlet_Type', explode(col('Outlet_Type'))) # explode a array
df.withColumn('Outlet_Type', split(col('Outlet_Type'), ' ')[1]) # puxa apenas o index 1
df.withColumn('Type1_flag', array_contains(col('Outlet_Type', 'Type1'))) # pega os valores unicos de uma array, e retorna valor booelan

In [0]:
df.groupby("Item_Type").agg(sum("Item_MRP")) #mesma lógica clássica do groupby do sql/pandas
df.groupby('Item_Type','Outlet_Size').agg(sum("Item_MRP"))


In [0]:
data = [('user1', 'book1'),
        ('user1', 'book2'),
        ('user2', 'book2'),
        ('user2', 'book4'),
        ('user3', 'book1')]
schema = 'user string, book string'
df_book = spark.createDataFrame(data, schema)


In [0]:
df_book.groupby('user').agg(collect_list('book')) # agrupa por usuário, e faz o agg a partir dos books

In [0]:
#exemplo de pivot
df.groupby('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP'))
#aqui, comparando com um pivot_table do pandas, é como se o Item_Type fosse o index, o Outlet_Size fosse o columns, e o Item_MRP fosse o values



In [0]:
df.withColumn('veg_flag', when(col('Item_Type') =='Meat','Non_Veg').otherwise('Veg'))
df.withColumn('veg_exp_flag', when((col('veg_flag') =='Veg') & (col('Item_MRP')<100),'Veg_inexpensive')\
    .when((col('veg_flag') =='Veg') & (col('Item_MRP')>100),'Veg_expensive')\
        .otherwise('Non_Veg'))


In [0]:
#aqui ele ensina os conceitos básicos dos joins, por ja saber bem, não colocarei aqui, apenas a sintaxe basica
df1.join(df2, df1['id'] == df2['id'], 'inner')


In [0]:
from pyspark.sql.window import Window

#nesse exemplo, conseguimos ver exatamente a quantidade de duplicadas de cada item, pois mesmo o id nao se repetindo, o Item_Identifier se repetiu
df.withColumn('id', row_number().over(Window.orderby('Item_Identifier')))

df.withColumn('rank', rank().over(Window.orderby('Item_Identifier'))) #igual no pandas e sql, a diferença para o dense rank ja sabemos

df.withColumn('sum', sum('Item_MRP').over(Window.partitionBy('Item_Type'))) #soma o Item_MRP por Item_Type
df.withColumn('sum', sum('Item_MRP').over(Window.orderby('Item_Type'))) #soma o Item_MRP de cada Item_Type, porem nao progressivamente, apenas o total. Diferente do partitionby, ele também agrega na soma os valores dos outros item_type
df.withColumn('sum', sum('Item_MRP').over(Window.orderby('Item_Type', 'Item_MRP').rowsBetween(Window.unboundedPreceding,Window.currentRow))) # aqui sim, ele soma progressivamente
df.withColumn('sum', sum('Item_MRP').over(Window.orderby('Item_Type', 'Item_MRP').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))) # aqui ele só mostra a soma total de todos os MRP



